# Art Institute of Chicago

### Quick set up and example

Establish a session, creat a cache, and fetch a random artwork.

In [58]:
import requests
from requests.adapters import HTTPAdapter, Retry
import os, json, hashlib, urllib.parse

# AIC Endpoint
BASE_URL = "https://api.artic.edu/api/v1/artworks"

# Create session with polite headers
session = requests.Session()
session.headers.update({
    "AIC-User-Agent": "AIC-sampling/0.1 (ex@mple.com)",
    "Accept": "application/json",
})
retries = Retry(
    total=3, backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET",)
)
session.mount("https://", HTTPAdapter(max_retries=retries))


# Cache
CACHE_DIR = "cache/aic"
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_key(url, params):
    q = urllib.parse.urlencode(sorted((params or {}).items()))
    key = f"{url}?{q}".encode("utf-8")
    return hashlib.sha1(key).hexdigest()

def _is_object_payload(p):
    return isinstance(p, dict) and isinstance(p.get("data"), dict) and "pagination" not in p

def _is_collection_payload(p):
    return isinstance(p, dict) and isinstance(p.get("data"), list)

# Cached fetch. Expect modifies behavior for collection/object
def fetch_json(url, params=None, cache_name=None, timeout=30, expect="any", force=False):
    # Prefer a cache name derived from the real request if none given
    if cache_name is None:
        cache_name = _cache_key(url, params)
    cache_path = os.path.join(CACHE_DIR, f"{cache_name}.json")

    def load_from_disk():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    def save_to_disk(data):
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    if not force and os.path.exists(cache_path):
        data = load_from_disk()
        if expect == "object" and _is_collection_payload(data):
            os.remove(cache_path)
        elif expect == "collection" and _is_object_payload(data):
            os.remove(cache_path)
        else:
            return data

    resp = session.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()
    save_to_disk(data)

    if expect == "object" and not _is_object_payload(data):
        raise ValueError(f"Expected an object payload at {url} but got collection/other.")
    if expect == "collection" and not _is_collection_payload(data):
        raise ValueError(f"Expected a collection payload at {url} but got object/other.")
    return data

# Example use
artwork_id = 131407 # https://www.artic.edu/artworks/131407/wine-cheese-and-fruit
detail_url = f"{BASE_URL}/{artwork_id}"
detail = fetch_json(detail_url, expect="object")
print(detail["data"].keys())
print("\nAll I want is:", detail["data"]["title"])
print("Some", (detail["data"].get("description") or "")[3:72], "please!")

dict_keys(['id', 'api_model', 'api_link', 'is_boosted', 'title', 'alt_titles', 'thumbnail', 'main_reference_number', 'has_not_been_viewed_much', 'boost_rank', 'date_start', 'date_end', 'date_display', 'date_qualifier_title', 'date_qualifier_id', 'artist_display', 'place_of_origin', 'description', 'short_description', 'dimensions', 'dimensions_detail', 'medium_display', 'inscriptions', 'credit_line', 'catalogue_display', 'publication_history', 'exhibition_history', 'provenance_text', 'edition', 'publishing_verification_level', 'internal_department_id', 'fiscal_year', 'fiscal_year_deaccession', 'is_public_domain', 'is_zoomable', 'max_zoom_window_size', 'copyright_notice', 'has_multimedia_resources', 'has_educational_resources', 'has_advanced_imaging', 'colorfulness', 'color', 'latitude', 'longitude', 'latlon', 'is_on_view', 'on_loan_display', 'gallery_title', 'gallery_id', 'nomisma_id', 'artwork_type_title', 'artwork_type_id', 'department_title', 'department_id', 'artist_id', 'artist_tit

### Querying for paintings

Searching for paintings that are public domain. Looking at the first record and seeing possibly interesting attributes.


In [59]:
QUERY = "painting"
search_url = f"{BASE_URL}/search"
search_params = {
    "q": QUERY,
    "query[term][is_public_domain]": "true",
}
search = fetch_json(search_url, params=search_params, expect="collection")
print("Querying for public domain paintings.", search["pagination"]["total"], "results.")

first_result = search["data"][0]
artwork_id = first_result["id"]

# 3) Fetch detail for first result (again expect object)
detail_url = f"{BASE_URL}/{artwork_id}"
detail = fetch_json(detail_url, expect="object")
print("All details from:", detail["data"]["title"])
print("Number of attributes:", len(detail["data"].keys()))
print({k for k in detail["data"].keys() if "_id" not in k and "api" not in k})

Querying for public domain paintings. 41269 results.
All details from: Painting with Troika
Number of attributes: 98
{'has_not_been_viewed_much', 'has_multimedia_resources', 'date_start', 'latitude', 'dimensions_detail', 'inscriptions', 'subject_titles', 'style_title', 'catalogue_display', 'latlon', 'dimensions', 'description', 'publishing_verification_level', 'main_reference_number', 'artwork_type_title', 'section_titles', 'title', 'fiscal_year', 'edition', 'thumbnail', 'id', 'is_on_view', 'style_titles', 'alt_titles', 'artist_title', 'artist_titles', 'theme_titles', 'timestamp', 'is_boosted', 'publication_history', 'is_zoomable', 'is_public_domain', 'date_qualifier_title', 'longitude', 'source_updated_at', 'copyright_notice', 'short_description', 'department_title', 'suggest_autocomplete_all', 'credit_line', 'term_titles', 'material_titles', 'medium_display', 'colorfulness', 'category_titles', 'technique_titles', 'max_zoom_window_size', 'artist_display', 'gallery_title', 'fiscal_year

### Getting Images

Sometimes images are just not there, but there might be an alternative image id on the detail. If the image is not there, it should be at least, flagged.

In [53]:
def choose_working_iiif_url(detail, sizes = [843, 600, 400]):
    iiif_base = detail["config"]["iiif_url"].rstrip("/")
    data = detail["data"]

    # Get image ID and alt IDs (if any)
    ids = []
    if data.get("image_id"):
        ids.append(data["image_id"])
    ids.extend(data.get("alt_image_ids", []) or [])

    tried = []
    for img_id in ids:
        for sz in sizes:
            url = f"{iiif_base}/{img_id}/full/{sz},/0/default.jpg"
            tried.append(url)
            try:
                h = session.head(url, timeout=8, allow_redirects=True)
                if h.status_code == 200:
                    return url, tried
                if h.status_code in (403, 404, 405):
                    g = session.get(url, stream=True, timeout=8)
                    if g.status_code == 200:
                        g.close()
                        return url, tried
                    g.close()
            except requests.RequestException:
                pass
    return None, tried

artwork_id  = search["data"][0]["id"]
detail = fetch_json(f"{BASE_URL}/{artwork_id}", cache_name=f"artwork_{artwork_id}")

best_url, tried = choose_working_iiif_url(detail)
print("Image from:", detail["data"]["title"])
print(best_url or "For some reason no URL is working")
if not best_url:
    print("\nTried:")
    for u in tried:
        print("  ", u)

artwork_id = search["data"][1]["id"]
detail = fetch_json(f"{BASE_URL}/{artwork_id}", cache_name=f"artwork_{artwork_id}")

best_url, tried = choose_working_iiif_url(detail)
print("Image from:", detail["data"]["title"])
print(best_url or "First URL not working")

Image from: Painting with Troika
For some reason no URL is working

Tried:
   https://www.artic.edu/iiif/2/a45e5f55-d02b-ce98-8bab-3af549684f58/full/843,/0/default.jpg
   https://www.artic.edu/iiif/2/a45e5f55-d02b-ce98-8bab-3af549684f58/full/600,/0/default.jpg
   https://www.artic.edu/iiif/2/a45e5f55-d02b-ce98-8bab-3af549684f58/full/400,/0/default.jpg
   https://www.artic.edu/iiif/2/9be68d8f-4854-268d-4df5-d668b6f494b3/full/843,/0/default.jpg
   https://www.artic.edu/iiif/2/9be68d8f-4854-268d-4df5-d668b6f494b3/full/600,/0/default.jpg
   https://www.artic.edu/iiif/2/9be68d8f-4854-268d-4df5-d668b6f494b3/full/400,/0/default.jpg
Image from: Painting with Green Center
https://www.artic.edu/iiif/2/c68f33ec-feb1-5277-334b-b71ac15ae394/full/843,/0/default.jpg


### Get one full record

In [55]:
import re
import pandas as pd

# Helper Functions:

# Remove some HTML tags in data
def html_to_text(s):
    if s is None:
        return None

    s = re.sub(r"<br\s*/?>", "\n", s, flags=re.I)
    s = re.sub(r"</p\s*>", "\n\n", s, flags=re.I)
    s = re.sub(r"<.*?>", "", s)
    return re.sub(r"\n{3,}", "\n\n", s).strip()

# Join elements in a list with ",". For style, technique, etc.
def list_to_str(x):
    if x is None:
        return None
    if isinstance(x, list):
        return ", ".join(str(v) for v in x if v is not None)
    return x

# Reset the id to 0
artwork_id = search["data"][0]["id"]
detail = fetch_json(f"{BASE_URL}/{artwork_id}", cache_name=f"artwork_{artwork_id}")
d = detail["data"]
print("Image",choose_working_iiif_url(detail))

row = {
    "api_link": d.get("api_link"),
    "title": d.get("title"),
    "has_not_been_viewed_much": d.get("has_not_been_viewed_much"),
    "date_start": d.get("date_start"),
    "date_end": d.get("date_end"),
    "date_display": d.get("date_display"),
    "artist_display": html_to_text(d.get("artist_display")),
    "place_of_origin": d.get("place_of_origin"),
    "description": html_to_text(d.get("description")),
    "short_description": html_to_text(d.get("short_description")),
    "dimensions": d.get("dimensions"),
    "dimensions_detail": json.dumps(d.get("dimensions_detail"), ensure_ascii=False),
    "medium_display": d.get("medium_display"),
    "inscriptions": d.get("inscriptions"),
    "publication_history": html_to_text(d.get("publication_history")),
    "exhibition_history": html_to_text(d.get("exhibition_history")),
    "provenance_text": html_to_text(d.get("provenance_text")),
    "copyright_notice": d.get("copyright_notice"),
    "is_on_view": d.get("is_on_view"),
    "artwork_type_title": d.get("artwork_type_title"),
    "artist_titles": list_to_str(d.get("artist_titles")),
    "category_titles": list_to_str(d.get("category_titles")),
    "term_titles": list_to_str(d.get("term_titles")),
    "style_titles": list_to_str(d.get("style_titles")),
    "classification_titles": list_to_str(d.get("classification_titles")),
    "subject_titles": list_to_str(d.get("subject_titles")),
    "material_titles": list_to_str(d.get("material_titles")),
    "technique_titles": list_to_str(d.get("technique_titles")),
    
    # Less relevant categories
    "credit_line": d.get("credit_line"),
    "gallery_title": d.get("gallery_title"),
    "is_public_domain": d.get("is_public_domain"),
    "is_zoomable": d.get("is_zoomable"),
    "updated_at": d.get("updated_at"),
    "source_updated_at": d.get("source_updated_at"),
    "image_url": choose_working_iiif_url(detail)[0],
}

df_one = pd.DataFrame([row])
display(df_one.head(1))

Image (None, ['https://www.artic.edu/iiif/2/a45e5f55-d02b-ce98-8bab-3af549684f58/full/843,/0/default.jpg', 'https://www.artic.edu/iiif/2/a45e5f55-d02b-ce98-8bab-3af549684f58/full/600,/0/default.jpg', 'https://www.artic.edu/iiif/2/a45e5f55-d02b-ce98-8bab-3af549684f58/full/400,/0/default.jpg', 'https://www.artic.edu/iiif/2/9be68d8f-4854-268d-4df5-d668b6f494b3/full/843,/0/default.jpg', 'https://www.artic.edu/iiif/2/9be68d8f-4854-268d-4df5-d668b6f494b3/full/600,/0/default.jpg', 'https://www.artic.edu/iiif/2/9be68d8f-4854-268d-4df5-d668b6f494b3/full/400,/0/default.jpg'])


,api_link,title,has_not_been_viewed_much,date_start,date_end,date_display,artist_display,place_of_origin,description,short_description,...,subject_titles,material_titles,technique_titles,credit_line,gallery_title,is_public_domain,is_zoomable,updated_at,source_updated_at,image_url
0,https://api.artic.edu/api/v1/artworks/8983,Painting with Troika,False,1911,1911,"January 18, 1911",Vasily Kandinsky\nBorn Moscow (formerly Russia...,Germany,"Vasily Kandinsky, along with Franz Marc, Gabr...",Vasily Kandinsky was a founding member of Blau...,...,"troikas, people, landscapes, animals, oxen, carts",,,Arthur Jerome Eddy Memorial Collection,Gallery 392,True,True,2025-09-10T12:00:36-05:00,2025-08-29T16:39:40-05:00,None


### Sampling multiple records from the paintings subset

In [60]:
INTERESTING_FIELDS = [
    "api_link", "title", "has_not_been_viewed_much",
    "date_start", "date_end", "date_display",
    "artist_display", "place_of_origin", "description", "short_description",
    "dimensions", "dimensions_detail", "medium_display", "inscriptions",
    "publication_history", "exhibition_history", "provenance_text",
    "copyright_notice", "is_on_view", "artwork_type_title",
    "artist_titles", "category_titles", "term_titles",
    "style_titles", "classification_titles", "subject_titles",
    "material_titles", "technique_titles"
]

EXTRA_FIELDS = ["credit_line", "gallery_title", "is_public_domain", "is_zoomable", "updated_at", "source_updated_at"]

def flatten_one(detail):
    d = detail["data"]
    row = {}
    row["api_link"] = d.get("api_link")
    row["title"] = d.get("title")
    row["has_not_been_viewed_much"] = d.get("has_not_been_viewed_much")
    row["date_start"] = d.get("date_start")
    row["date_end"] = d.get("date_end")
    row["date_display"] = d.get("date_display")
    row["artist_display"] = html_to_text(d.get("artist_display"))
    row["place_of_origin"] = d.get("place_of_origin")
    row["description"] = html_to_text(d.get("description"))
    row["short_description"] = html_to_text(d.get("short_description"))
    row["dimensions"] = d.get("dimensions")
    row["dimensions_detail"] = json.dumps(d.get("dimensions_detail"), ensure_ascii=False)
    row["medium_display"] = d.get("medium_display")
    row["inscriptions"] = d.get("inscriptions")
    row["publication_history"] = html_to_text(d.get("publication_history"))
    row["exhibition_history"] = html_to_text(d.get("exhibition_history"))
    row["provenance_text"] = html_to_text(d.get("provenance_text"))
    row["copyright_notice"] = d.get("copyright_notice")
    row["is_on_view"] = d.get("is_on_view")
    row["artwork_type_title"] = d.get("artwork_type_title")
    row["artist_titles"] = list_to_str(d.get("artist_titles"))
    row["category_titles"] = list_to_str(d.get("category_titles"))
    row["term_titles"] = list_to_str(d.get("term_titles"))
    row["style_titles"] = list_to_str(d.get("style_titles"))
    row["classification_titles"] = list_to_str(d.get("classification_titles"))
    row["subject_titles"] = list_to_str(d.get("subject_titles"))
    row["material_titles"] = list_to_str(d.get("material_titles"))
    row["technique_titles"] = list_to_str(d.get("technique_titles"))


    row["image_url"] = choose_working_iiif_url(detail)[0]

    for f in EXTRA_FIELDS:
        row[f] = d.get(f)

    return row

In [57]:
# Parameters
TARGET_N = 100
PER_PAGE = 25
MAX_PAGES = 200
SEARCH_Q = "painting"

rows = []
ids_seen = set()

def search_page(page, per_page = PER_PAGE):
    url = f"{BASE_URL}/search"
    params = {
        "q": SEARCH_Q,
        "query[term][is_public_domain]": "true",
        "limit": per_page,
        "page": page,
        "fields": "id,title,artwork_type_title,is_public_domain,image_id,alt_image_ids,is_zoomable"
    }
    return fetch_json(url, params=params, cache_name=f"search_pd_painting_page_{page}")

page = 1
while len(rows) < TARGET_N and page <= MAX_PAGES:
    try:
        res = search_page(page)
    except Exception as e:
        print(f"[warn] search page {page} failed: {e}")
        page += 1
        continue

    data = res.get("data", []) or []
    if not data:
        print(f"[info] no data on page {page}; stopping.")
        break

    # prefilter: make it more likely images will work & type is Painting
    candidates = [
        d for d in data
        if (d.get("artwork_type_title") == "Painting")
        and (d.get("image_id") or (d.get("alt_image_ids") or []))
    ]

    for stub in candidates:
        if len(rows) >= TARGET_N:
            break
        art_id = stub["id"]
        if art_id in ids_seen:
            continue
        ids_seen.add(art_id)

        # fetch detail with cache and flatten
        try:
            detail_url = f"{BASE_URL}/{art_id}"
            detail = fetch_json(detail_url, cache_name=f"artwork_{art_id}")
            row = flatten_one(detail)

            # Ensure it's still a painting and public domain (defensive)
            if detail["data"].get("artwork_type_title") != "Painting":
                continue
            if not detail["data"].get("is_public_domain", False):
                continue
            if not row["image_url"]:
                continue

            rows.append(row)

        except Exception as e:
            print(f"[warn] detail {art_id} failed: {e}")
            continue

    # move to next page
    page += 1

# Build DataFrame
df = pd.DataFrame(rows)
print(f"Collected {len(df)} rows.")
df.head(3)

Collected 100 rows.


,api_link,title,has_not_been_viewed_much,date_start,date_end,date_display,artist_display,place_of_origin,description,short_description,...,subject_titles,material_titles,technique_titles,image_url,credit_line,gallery_title,is_public_domain,is_zoomable,updated_at,source_updated_at
0,https://api.artic.edu/api/v1/artworks/8987,Painting with Green Center,False,1913,1913,1913,Vasily Kandinsky\nBorn Moscow (formerly Russia...,Germany,None,None,...,"Century of Progress, world's fairs, Chicago Wo...",,,https://www.artic.edu/iiif/2/c68f33ec-feb1-527...,Arthur Jerome Eddy Memorial Collection,Gallery 392,True,True,2025-09-10T06:12:14-05:00,2025-04-03T13:25:34-05:00
1,https://api.artic.edu/api/v1/artworks/11,Self-Portrait,False,1878,1878,1878,"Walter Shirlaw\nAmerican, 1838–1909",United States,None,None,...,self-portraits,"oil paint (paint), organic material",,https://www.artic.edu/iiif/2/7b7a6f39-1cd8-ea2...,Gift of Joseph M. Rogers,None,True,True,2025-09-10T11:30:41-05:00,2023-12-07T14:12:37-06:00
2,https://api.artic.edu/api/v1/artworks/6002,The Print Collector,False,1855,1865,c. 1860,"Honoré Victorin Daumier (French, 1808–1879)",France,None,None,...,"prints, France, French, hat, jackets, man","oil paint (paint), paint, painting, panel (woo...","oil painting, cradling, painting, painting (im...",https://www.artic.edu/iiif/2/f1dce916-58a3-db5...,Gift of the Estate of Marshall Field,Gallery 225,True,True,2025-09-09T20:13:12-05:00,2025-08-04T14:05:43-05:00


In [80]:
out_csv = f"{CACHE_DIR}/aic_paintings_sample_100.csv"
df.to_csv(out_csv, index=False)

### Decorative Arts

In [ ]:
# Parameters
TARGET_N = 100
PER_PAGE = 25
MAX_PAGES = 200
SEARCH_Q = "decorative arts"

rows = []
ids_seen = set()

def search_page(page, per_page = PER_PAGE):
    url = f"{BASE_URL}/search"
    params = {
        "q": SEARCH_Q,
        "query[term][is_public_domain]": "true",
        "limit": per_page,
        "page": page,
        "fields": "id,title,artwork_type_title,is_public_domain,image_id,alt_image_ids,is_zoomable"
    }
    return fetch_json(url, params=params, cache_name=f"search_pd_decorative_arts_page_{page}")

page = 1
while len(rows) < TARGET_N and page <= MAX_PAGES:
    try:
        res = search_page(page)
    except Exception as e:
        print(f"[warn] search page {page} failed: {e}")
        page += 1
        continue

    data = res.get("data", []) or []
    if not data:
        print(f"[info] no data on page {page}; stopping.")
        break

    # prefilter: make it more likely images will work & type is Painting
    candidates = [
        d for d in data
        if (d.get("image_id") or (d.get("alt_image_ids") or []))
    ]

    for stub in candidates:
        if len(rows) >= TARGET_N:
            break
        art_id = stub["id"]
        if art_id in ids_seen:
            continue
        ids_seen.add(art_id)

        try:
            detail_url = f"{BASE_URL}/{art_id}"
            detail = fetch_json(detail_url, cache_name=f"artwork_{art_id}")
            row = flatten_one(detail)

            # Ensure it's still a painting and public domain
            if not detail["data"].get("is_public_domain", False):
                continue
            if not row["image_url"]:
                continue

            rows.append(row)

        except Exception as e:
            print(f"[warn] detail {art_id} failed: {e}")
            continue

    # move to next page
    page += 1

df = pd.DataFrame(rows)
print(f"Collected {len(df)} rows.")
df.head(3)

In [87]:
out_csv = f"{CACHE_DIR}/aic_art_sample_100.csv"
df.to_csv(out_csv, index=False)

### Every public object (paintings, sculptures, etc)

In [ ]:
# Parameters
TARGET_N = 1000
PER_PAGE = 25
MAX_PAGES = 200

rows = []
ids_seen = set()

def search_page(page, per_page = PER_PAGE):
    url = f"{BASE_URL}/search"
    params = {
        "query[term][is_public_domain]": "true",
        "limit": per_page,
        "page": page,
        "fields": "id,title,artwork_type_title,is_public_domain,image_id,alt_image_ids,is_zoomable"
    }
    return fetch_json(url, params=params, cache_name=f"search_pd_art_page_{page}")

page = 1
while len(rows) < TARGET_N and page <= MAX_PAGES:
    try:
        res = search_page(page)
    except Exception as e:
        print(f"[warn] search page {page} failed: {e}")
        page += 1
        continue

    data = res.get("data", []) or []
    if not data:
        print(f"[info] no data on page {page}; stopping.")
        break

    # prefilter: make it more likely images will work & type is Painting
    candidates = [
        d for d in data
        if (d.get("image_id") or (d.get("alt_image_ids") or []))
    ]

    for stub in candidates:
        if len(rows) >= TARGET_N:
            break
        art_id = stub["id"]
        if art_id in ids_seen:
            continue
        ids_seen.add(art_id)

        # fetch detail with cache and flatten
        try:
            detail_url = f"{BASE_URL}/{art_id}"
            detail = fetch_json(detail_url, cache_name=f"artwork_{art_id}")
            row = flatten_one(detail)

            if not detail["data"].get("is_public_domain", False):
                continue
            if not row["image_url"]:
                continue

            rows.append(row)

        except Exception as e:
            print(f"[warn] detail {art_id} failed: {e}")
            continue

    # move to next page
    page += 1

# Build DataFrame
df = pd.DataFrame(rows)
print(f"Collected {len(df)} rows.")
df.head(3)